# HR Attrition Analysis
### HRDataset v13 | Huebner & Patalano | Retention ROI Edition

  
**Stack:** Python (pandas + DuckDB) → Power BI  
**Dataset:** HRDataset v13 from Kaggle (version 2) — 4 linked CSV files  
**Objective:** Quantify attrition cost, identify at-risk employees, and model retention ROI

---

| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Data Loading |
| 3 | Data Cleaning |
| 4 | Data Audit |
| 5 | Exploratory Analysis |
| 6 | Master Table Build |
| 7 | Cost Engine |
| 8 | Risk Segmentation & Export |


---
## Section 1 — Setup & Imports

Libraries used:
- **pandas** — data loading, structural checks, and cleaning
- **duckdb** — SQL queries directly on pandas dataframes; analytical heavy lifting


In [1]:
import pandas as pd
import duckdb

---
## Section 2 — Data Loading

Four CSV files from HRDataset v13 (Kaggle version 2):

| File | Role | Notes |
|------|------|-------|
| `HRDataset_v13.csv` | Fact table — 401 raw rows (91 blank rows removed in cleaning) | Core employee data |
| `production_staff.csv` | Supplement — Production dept performance metrics | 256 raw rows (47 blank rows removed) |
| `recruiting_costs.csv` | Dimension — actual spend per source per month | Wide format → must be unpivoted |
| `salary_grid.csv` | Dimension — hourly and annual pay bands per position | Covers 12 of 32 positions |


In [2]:
# ── Main HR dataset ──────────────────────────────────────────────────────────
df = pd.read_csv(r'C:\Ongoing course\Projects\HR Attrition Analytics\Raw\HRDataset_v13.csv')
df.tail()

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,PayRate,...,Department,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30
396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
397,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# ── Production staff (performance metrics for Production dept only) ──────────
prod = pd.read_csv(r'C:\Ongoing course\Projects\HR Attrition Analytics\Raw\production_staff.csv')
prod.tail()

,Employee Name,Race Desc,Date of Hire,TermDate,Reason for Term,Employment Status,Department,Position,Pay,Manager Name,Performance Score,Abutments/Hour Wk 1,Abutments/Hour Wk 2,Daily Error Rate,90-day Complaints
251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
253,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
255,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ── Recruiting costs — wide format (sources × months) ────────────────────────
recruit = pd.read_csv(r'C:\Ongoing course\Projects\HR Attrition Analytics\Raw\recruiting_costs.csv')
recruit.head()

,Employment Source,January,February,March,April,May,June,July,August,September,October,November,December,Total
0,Billboard,520,520,520,520,0,0,612,612,729,749,910,500,6192
1,Careerbuilder,410,410,410,820,820,410,410,820,820,1230,820,410,7790
2,Company Intranet - Partner,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Diversity Job Fair,0,5129,0,0,0,0,0,4892,0,0,0,0,10021
4,Employee Referral,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# ── Salary grid — header=1 skips the merged top row ──────────────────────────
# Covers 12 positions: Production Technician I/II, Lead Production Technician,
# Accountant I/II, Sr. Accountant, Network Engineer, Sr. Network Engineer,
# Database Administrator, Sr. DBA, Administrative Assistant, Sr. Administrative Assistant
salary = pd.read_csv(r'C:\Ongoing course\Projects\HR Attrition Analytics\Raw\salary_grid.csv', header=1)
salary.head(10)

,Position,Min,Mid,Max,Min.1,Mid.1,Max.1,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,Administrative Assistant,30000.0,40000.0,50000.0,14.42,19.23,24.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sr. Administrative Assistant,35000.0,45000.0,55000.0,16.83,21.63,26.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Accountant I,42274.0,51425.0,62299.0,20.32,24.72,29.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Accountant II,50490.0,62158.0,74658.0,24.27,29.88,35.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sr. Accountant,63264.0,76988.0,92454.0,30.42,37.01,44.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Network Engineer,50845.0,66850.0,88279.0,24.44,32.14,42.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sr. Network Engineer,79428.0,99458.0,120451.0,38.19,47.82,57.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Database Administrator,50569.0,68306.0,93312.0,24.31,32.84,44.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Sr. DBA,92863.0,116007.0,139170.0,44.65,55.77,66.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Production Technician I,30000.0,40000.0,50000.0,14.42,19.23,24.04,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN


---
## Section 3 — Data Cleaning

Steps applied to each file before any analysis:

1. **Remove blank rows** — HRDataset_v13 has 91 trailing blank rows; production_staff has 47
2. **Strip trailing whitespace** — Department and Position columns had trailing spaces that break WHERE filters
3. **Fix recruiting costs** — Drop Total column, then unpivot from wide to long format
4. **Fix salary grid** — Drop unnamed/empty columns, rename to consistent column names
5. **Null handling** — TermReason (1 null → Unknown), ManagerID (8 nulls → Webster Butler = 39)
6. **Date conversion** — 4 columns stored as object dtype; must be datetime for tenure calculations\n7. **DOB century fix** — 101 employees born 1919–1976 had two-digit birth years misparsed as 2019–2076 by the date parser. Any DOB after the data snapshot date (2019-02-28) is impossible — subtract 100 years to correct. Without this fix, one third of the workforce appears to have negative age and the entire age analysis is invalid.


In [6]:
# ── Remove blank rows ─────────────────────────────────────────────────────────
df = df.dropna(how='all')
prod = prod.dropna(how='all')

print("HR dataset shape after cleaning:", df.shape)      
print("Production staff shape after cleaning:", prod.shape)  

HR dataset shape after cleaning: (310, 35)
Production staff shape after cleaning: (209, 15)


In [7]:
# ── Strip trailing whitespace from all string columns ─────────────────────────
# Fixes: 'Production       ' vs 'Production', 'Data Analyst ' vs 'Data Analyst'
df[df.select_dtypes(include='object').columns] = (
    df.select_dtypes(include='object').apply(lambda x: x.str.strip())
)

In [8]:
# ── Recruiting costs: drop Total column, then unpivot wide → long ────────────
# Before melt: 22 sources × 14 columns (12 months + Employment Source + Total)
# After melt:  264 rows × 3 columns (Employment Source, Month, Cost)
recruit = recruit.drop("Total", axis=1)
recruit = pd.melt(recruit, id_vars='Employment Source', var_name='Month', value_name='Cost')
recruit = recruit.rename(columns={'Employment Source': 'EmploymentSource'})

print("Recruiting costs shape after melt:", recruit.shape)  
recruit.head()

Recruiting costs shape after melt: (264, 3)


,EmploymentSource,Month,Cost
0,Billboard,January,520
1,Careerbuilder,January,410
2,Company Intranet - Partner,January,0
3,Diversity Job Fair,January,0
4,Employee Referral,January,0


In [9]:
# ── Salary grid: clean unnamed columns and rename ────────────────────────────
salary = salary.dropna(axis=1, how='all')
salary = salary.dropna(how='all')
salary = salary.drop("Unnamed: 9", axis=1)
salary = salary.rename(columns={
    "Min":   "AnnualMin",
    "Mid":   "AnnualMid",
    "Max":   "AnnualMax",
    "Min.1": "HourlyMin",
    "Mid.1": "HourlyMid",
    "Max.1": "HourlyMax"
})
salary.head()

,Position,AnnualMin,AnnualMid,AnnualMax,HourlyMin,HourlyMid,HourlyMax
0,Administrative Assistant,30000.0,40000.0,50000.0,14.42,19.23,24.04
1,Sr. Administrative Assistant,35000.0,45000.0,55000.0,16.83,21.63,26.44
2,Accountant I,42274.0,51425.0,62299.0,20.32,24.72,29.95
3,Accountant II,50490.0,62158.0,74658.0,24.27,29.88,35.89
4,Sr. Accountant,63264.0,76988.0,92454.0,30.42,37.01,44.45


In [10]:
# ── Null handling ────────────────────────────────────────────────────────────
# TermReason: 1 null (terminated employee, reason unknown) → 'Unknown'
df['TermReason'] = df['TermReason'].fillna('Unknown')

# ManagerID: 8 nulls (all belong to Webster Butler, confirmed ManagerID = 39)
df['ManagerID'] = df['ManagerID'].fillna(39).astype(int)

In [11]:
# ── Date column conversion (object → datetime) ───────────────────────────────
# Required for DATEDIFF calculations in tenure and age analysis
df["DateofHire"]                = pd.to_datetime(df["DateofHire"])
df["DateofTermination"]         = pd.to_datetime(df["DateofTermination"])
df["DOB"]                       = pd.to_datetime(df["DOB"])
df["LastPerformanceReview_Date"] = pd.to_datetime(df["LastPerformanceReview_Date"])

print("Date columns converted successfully")

Date columns converted successfully


C:\Users\rupsa\AppData\Local\Temp\ipykernel_11884\1870548710.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DateofTermination"]         = pd.to_datetime(df["DateofTermination"])
C:\Users\rupsa\AppData\Local\Temp\ipykernel_11884\1870548710.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DOB"]                       = pd.to_datetime(df["DOB"])


In [12]:
# ── DOB century fix (CRITICAL DATA QUALITY ISSUE) ────────────────────────────
# DISCOVERED DURING DASHBOARD QA: 101 employees have impossible future birth dates.
# Root cause: two-digit birth years ("10/23/64") were parsed as 2064 instead of 1964.
# Impact: one third of the workforce appeared to have negative age — the original
# "Under 25 = 98 employees" finding was entirely misparsed dates, not young employees.
# Fix: any DOB after the data snapshot date (2019-02-28) is impossible → subtract 100 years.

mask = df['DOB'] > pd.Timestamp('2019-02-28')
df.loc[mask, 'DOB'] = df.loc[mask, 'DOB'] - pd.DateOffset(years=100)

print(f"Fixed {mask.sum()} misparsed DOBs")
print(f"DOB range after fix: {df['DOB'].min().date()} to {df['DOB'].max().date()}")

Fixed 100 misparsed DOBs
DOB range after fix: 1951-01-02 to 1992-08-17


---
## Section 4 — Data Audit

Five checks before any analysis to verify data quality:

| Check | Purpose |
|-------|---------|
| 4.1 Shape & nulls | Confirm row count and identify missing values |
| 4.2 Duplicate detection | Rows, EmpIDs, employee names |
| 4.3 Termd consistency | Active employees must have no exit date; terminated must have one |
| 4.4 Range validation | Scored columns must be within expected bounds |
| 4.5 Date sequence | Termination date must be after hire date |

**Key findings:** Date columns stored as object (fixed in Section 3). ManagerID nulls all traced 
to Webster Butler (fixed). DaysLateLast30 is 0 for all active employees — no analytical value, 
removed from flight risk score. LastPerformanceReview_Date and DaysLateLast30 are NULL for all 
103 terminated employees — scores captured only for active employees in 2019 review cycle. 
**CRITICAL: 101 employees had two-digit birth years misparsed as future dates (2019–2076) — 
discovered during dashboard QA, corrected with the DOB century fix in Section 3.**


In [13]:
# 4.1 — Shape, dtypes, and nulls
print("Shape:", df.shape)
print()
print(df.info())
print()
nulls = df.isnull().sum()
print("Columns with nulls:")
print(nulls[nulls > 0])

Shape: (310, 35)

<class 'pandas.core.frame.DataFrame'>
Index: 310 entries, 0 to 309
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Employee_Name               310 non-null    object        
 1   EmpID                       310 non-null    float64       
 2   MarriedID                   310 non-null    float64       
 3   MaritalStatusID             310 non-null    float64       
 4   GenderID                    310 non-null    float64       
 5   EmpStatusID                 310 non-null    float64       
 6   DeptID                      310 non-null    float64       
 7   PerfScoreID                 310 non-null    float64       
 8   FromDiversityJobFairID      310 non-null    float64       
 9   PayRate                     310 non-null    float64       
 10  Termd                       310 non-null    float64       
 11  PositionID                  310 non-null    f

In [14]:
# 4.2 — Duplicate detection
# HR-specific: check rows, EmpIDs, AND names (same name = potential rehire)
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate EmpIDs:", df.duplicated(subset='EmpID').sum())

name_counts = df['Employee_Name'].value_counts()
print("\nNames appearing more than once:")
print(name_counts[name_counts > 1])

Fully duplicate rows: 0
Duplicate EmpIDs: 0

Names appearing more than once:
Series([], Name: count, dtype: int64)


In [15]:
# 4.3 — Termd vs DateofTermination consistency
# Clean result: only two rows should appear
# Termd=0 / No date (active employees, expected)
# Termd=1 / Has date (terminated employees, expected)
duckdb.sql("""
    SELECT
        Termd,
        CASE WHEN DateofTermination IS NULL THEN 'No date' ELSE 'Has date' END AS date_status,
        COUNT(*) AS count
    FROM df
    GROUP BY Termd, date_status
    ORDER BY Termd, date_status
""").show()

┌────────┬─────────────┬───────┐
│ Termd  │ date_status │ count │
│ double │   varchar   │ int64 │
├────────┼─────────────┼───────┤
│    0.0 │ No date     │   207 │
│    1.0 │ Has date    │   103 │
└────────┴─────────────┴───────┘



In [16]:
# 4.4 — Range validation on all analytical columns
print("=== Engagement Survey (expected 1-5) ===")
print(df['EngagementSurvey'].describe())

print("\n=== Employee Satisfaction (expected 1-5) ===")
print(df['EmpSatisfaction'].describe())

print("\n=== PayRate (expected positive) ===")
print(df['PayRate'].describe())

print("\n=== Days Late Last 30 (expected >= 0) ===")
print(df['DaysLateLast30'].describe())

print("\n=== Special Projects Count (expected >= 0) ===")
print(df['SpecialProjectsCount'].describe())

=== Engagement Survey (expected 1-5) ===
count    310.000000
mean       3.332097
std        1.290590
min        1.030000
25%        2.082500
50%        3.470000
75%        4.520000
max        5.000000
Name: EngagementSurvey, dtype: float64

=== Employee Satisfaction (expected 1-5) ===
count    310.000000
mean       3.890323
std        0.910690
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: EmpSatisfaction, dtype: float64

=== PayRate (expected positive) ===
count    310.000000
mean      31.284806
std       15.383615
min       14.000000
25%       20.000000
50%       24.000000
75%       45.315000
max       80.000000
Name: PayRate, dtype: float64

=== Days Late Last 30 (expected >= 0) ===
count    207.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: DaysLateLast30, dtype: float64

=== Special Projects Count (expected >= 0) ===
count    310.000000
mean       1.209677
s

In [17]:
# DuckDB range check — min/max across key columns in one query
duckdb.sql("""
    SELECT
        MIN(EngagementSurvey) AS eng_min, MAX(EngagementSurvey) AS eng_max,
        MIN(EmpSatisfaction)  AS sat_min, MAX(EmpSatisfaction)  AS sat_max,
        MIN(PayRate)          AS sal_min, MAX(PayRate)          AS sal_max,
        MIN(DaysLateLast30)   AS late_min, MAX(DaysLateLast30)  AS late_max
    FROM df
""").show()

┌─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┬──────────┐
│ eng_min │ eng_max │ sat_min │ sat_max │ sal_min │ sal_max │ late_min │ late_max │
│ double  │ double  │ double  │ double  │ double  │ double  │  double  │  double  │
├─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼──────────┼──────────┤
│    1.03 │     5.0 │     1.0 │     5.0 │    14.0 │    80.0 │      0.0 │      0.0 │
└─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴──────────┴──────────┘



In [18]:
# 4.5 — Date sequence check (no one terminated before they were hired)
duckdb.sql("""
    SELECT COUNT(*) AS bad_date_sequence
    FROM df
    WHERE Termd = 1
    AND DateofTermination < DateofHire
""").show()

┌───────────────────┐
│ bad_date_sequence │
│       int64       │
├───────────────────┤
│                 0 │
└───────────────────┘



---
## Section 5 — Exploratory Analysis

All EDA queries for Power BI Pages 1 and 2. Key findings documented inline.

| Sub-section | Covers |
|-------------|--------|
| 5.1 | TermReason & EmploymentStatus breakdown |
| 5.2 | Department attrition breakdown |
| 5.3 | Overall attrition rates |
| 5.4 | Engagement & satisfaction analysis |
| 5.5 | Performance score distribution |
| 5.6 | Tenure at exit analysis |
| 5.7 | Demographics (gender, age) |
| 5.8 | Manager-level attrition |


### 5.1 — TermReason & EmploymentStatus Breakdown

In [19]:
# All unique termination reasons grouped by employment status
# KEY FINDING: 'attendance' and 'performance' appear under BOTH Voluntary and Involuntary
# → TermType derived from EmploymentStatus column, not TermReason string matching
duckdb.sql("""
    SELECT EmploymentStatus, TermReason, COUNT(*) AS count
    FROM df
    GROUP BY EmploymentStatus, TermReason
    ORDER BY EmploymentStatus, count DESC
""").show(max_rows=50)

┌────────────────────────┬──────────────────────────────────┬───────┐
│    EmploymentStatus    │            TermReason            │ count │
│        varchar         │             varchar              │ int64 │
├────────────────────────┼──────────────────────────────────┼───────┤
│ Active                 │ N/A - still employed             │   182 │
│ Future Start           │ N/A - Has not started yet        │    11 │
│ Leave of Absence       │ N/A - still employed             │    14 │
│ Terminated for Cause   │ attendance                       │     6 │
│ Terminated for Cause   │ no-call, no-show                 │     3 │
│ Terminated for Cause   │ performance                      │     3 │
│ Terminated for Cause   │ Unknown                          │     1 │
│ Terminated for Cause   │ gross misconduct                 │     1 │
│ Terminated for Cause   │ hours                            │     1 │
│ Voluntarily Terminated │ Another position                 │    20 │
│ Voluntarily Termin

### 5.2 — Department Breakdown

In [20]:
# Department headcount with voluntary/involuntary split
# NOTE: Software Engineering (11), Admin Offices (10), Executive Office (1)
# merged into 'Other' in dashboard — sample too small for reliable insights
duckdb.sql("""
    SELECT Department,
           COUNT(*) AS total_employees,
           COUNT(CASE WHEN Termd=0 THEN EmpID END) AS active_employees,
           COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END) AS invol_terminated,
           COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END) AS vol_terminated,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END)/COUNT(*)*100, 2) AS pct_invol,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END)/COUNT(*)*100, 2) AS pct_vol
    FROM df
    GROUP BY Department
    ORDER BY total_employees DESC
""")

┌──────────────────────┬─────────────────┬──────────────────┬──────────────────┬────────────────┬───────────┬─────────┐
│      Department      │ total_employees │ active_employees │ invol_terminated │ vol_terminated │ pct_invol │ pct_vol │
│       varchar        │      int64      │      int64       │      int64       │     int64      │  double   │ double  │
├──────────────────────┼─────────────────┼──────────────────┼──────────────────┼────────────────┼───────────┼─────────┤
│ Production           │             208 │              125 │                8 │             75 │      3.85 │   36.06 │
│ IT/IS                │              50 │               40 │                4 │              6 │       8.0 │    12.0 │
│ Sales                │              31 │               27 │                1 │              3 │      3.23 │    9.68 │
│ Admin Offices        │              10 │                7 │                1 │              2 │      10.0 │    20.0 │
│ Software Engineering │              10

In [21]:
# Production voluntary exit reasons — avoidable vs unavoidable split
# KEY FINDING: 31 of 75 voluntary exits avoidable (Unhappy 14 + Money 11 + Hours 6)
duckdb.sql("""
    SELECT Department, TermReason, COUNT(*) AS count
    FROM df
    WHERE Department='Production' AND EmploymentStatus='Voluntarily Terminated'
    GROUP BY Department, TermReason
    ORDER BY count DESC
""")

┌────────────┬──────────────────────────────────┬───────┐
│ Department │            TermReason            │ count │
│  varchar   │             varchar              │ int64 │
├────────────┼──────────────────────────────────┼───────┤
│ Production │ Another position                 │    17 │
│ Production │ unhappy                          │    14 │
│ Production │ more money                       │    11 │
│ Production │ hours                            │     6 │
│ Production │ career change                    │     6 │
│ Production │ return to school                 │     5 │
│ Production │ relocation out of area           │     4 │
│ Production │ military                         │     4 │
│ Production │ retiring                         │     4 │
│ Production │ maternity leave - did not return │     2 │
│ Production │ attendance                       │     1 │
│ Production │ medical issues                   │     1 │
└────────────┴──────────────────────────────────┴───────┘
  12 rows     

### 5.3 — Overall Attrition Rates

In [22]:
# Overall, voluntary, and involuntary attrition rates
# Future Start (11 employees) excluded — never actually employed
# KEY FINDING: 34.45% overall, 29.43% voluntary, 5.02% involuntary (6:1 ratio)
duckdb.sql("""
    SELECT
        COUNT(EmpID) AS total,
        COUNT(CASE WHEN Termd=1 THEN EmpID END) AS terminated,
        COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END) AS vol_terminated,
        COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END) AS invol_terminated,
        ROUND(COUNT(CASE WHEN Termd=1 THEN EmpID END)/COUNT(EmpID)*100, 2) AS overall_attrition_rate,
        ROUND(COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END)/COUNT(EmpID)*100, 2) AS vol_attrition_rate,
        ROUND(COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END)/COUNT(EmpID)*100, 2) AS invol_attrition_rate
    FROM df
    WHERE EmploymentStatus != 'Future Start'
""").show()

┌───────┬────────────┬────────────────┬──────────────────┬────────────────────────┬────────────────────┬──────────────────────┐
│ total │ terminated │ vol_terminated │ invol_terminated │ overall_attrition_rate │ vol_attrition_rate │ invol_attrition_rate │
│ int64 │   int64    │     int64      │      int64       │         double         │       double       │        double        │
├───────┼────────────┼────────────────┼──────────────────┼────────────────────────┼────────────────────┼──────────────────────┤
│   299 │        103 │             88 │               15 │                  34.45 │              29.43 │                 5.02 │
└───────┴────────────┴────────────────┴──────────────────┴────────────────────────┴────────────────────┴──────────────────────┘



### 5.4 — Engagement & Satisfaction Analysis

In [23]:
# Engagement/satisfaction: leavers vs stayers
# LIMITATION: Scores captured only in Jan-Feb 2019 (active employees only).
# Terminated employees have NULL review dates — comparison across groups is invalid.
# These scores are used for active employee analysis only (flight risk scoring).
duckdb.sql("""
    SELECT Termd,
           ROUND(AVG(EngagementSurvey), 2) AS avg_engagement,
           ROUND(AVG(EmpSatisfaction), 2) AS avg_satisfaction
    FROM df
    WHERE EmploymentStatus != 'Future Start'
    GROUP BY Termd
""")

┌────────┬────────────────┬──────────────────┐
│ Termd  │ avg_engagement │ avg_satisfaction │
│ double │     double     │      double      │
├────────┼────────────────┼──────────────────┤
│    1.0 │           3.33 │             3.87 │
│    0.0 │           3.34 │              3.9 │
└────────┴────────────────┴──────────────────┘

In [24]:
# Engagement by department — active employees only
# NOTE: Only Production, IT/IS, and Sales have sufficient sample size (10+)
duckdb.sql("""
    SELECT Department,
           ROUND(AVG(EngagementSurvey), 2) AS avg_engagement,
           ROUND(AVG(EmpSatisfaction), 2) AS avg_satisfaction
    FROM df
    WHERE EmploymentStatus != 'Future Start' AND Termd = 0
    GROUP BY Department
    ORDER BY avg_engagement, avg_satisfaction
""")

┌──────────────────────┬────────────────┬──────────────────┐
│      Department      │ avg_engagement │ avg_satisfaction │
│       varchar        │     double     │      double      │
├──────────────────────┼────────────────┼──────────────────┤
│ Software Engineering │           3.07 │              4.0 │
│ Sales                │           3.22 │             3.92 │
│ IT/IS                │           3.24 │             4.03 │
│ Production           │           3.37 │             3.87 │
│ Admin Offices        │           3.98 │             3.57 │
│ Executive Office     │           4.83 │              3.0 │
└──────────────────────┴────────────────┴──────────────────┘

### 5.5 — Performance Score Distribution

In [25]:
# Performance distribution: leavers vs stayers (% within each group)
# KEY FINDING: 'Needs Improvement' is 3x higher in leavers (9.71% vs 3.57%)
# 'Exceeds' is nearly 2x higher in stayers (13.78% vs 7.77%)
# Manager-assigned scores — valid for leavers vs stayers comparison (unlike survey scores)
duckdb.sql("""
    SELECT Termd, PerformanceScore,
           COUNT(EmpID) AS count,
           ROUND(COUNT(EmpID)/SUM(COUNT(*)) OVER (PARTITION BY Termd)*100, 2) AS pct
    FROM df
    WHERE EmploymentStatus != 'Future Start'
    GROUP BY Termd, PerformanceScore
    ORDER BY Termd, pct DESC
""")

┌────────┬───────────────────┬───────┬────────┐
│ Termd  │ PerformanceScore  │ count │  pct   │
│ double │      varchar      │ int64 │ double │
├────────┼───────────────────┼───────┼────────┤
│    0.0 │ Fully Meets       │   154 │  78.57 │
│    0.0 │ Exceeds           │    27 │  13.78 │
│    0.0 │ PIP               │     8 │   4.08 │
│    0.0 │ Needs Improvement │     7 │   3.57 │
│    1.0 │ Fully Meets       │    81 │  78.64 │
│    1.0 │ Needs Improvement │    10 │   9.71 │
│    1.0 │ Exceeds           │     8 │   7.77 │
│    1.0 │ PIP               │     4 │   3.88 │
└────────┴───────────────────┴───────┴────────┘

### 5.6 — Tenure at Exit Analysis

In [26]:
# Tenure at exit — binned into bands
# KEY FINDING: 2-5yr band = 40.78% of exits (career stagnation zone)
#              0-1yr band  = 31.07% of exits (hiring or onboarding failure)
#              0-5yr combined = 94% of all exits
duckdb.sql("""
    SELECT
        CASE
            WHEN DATEDIFF('day', DateofHire, DateofTermination) / 365.25 < 1  THEN '0-1 yr'
            WHEN DATEDIFF('day', DateofHire, DateofTermination) / 365.25 < 2  THEN '1-2 yr'
            WHEN DATEDIFF('day', DateofHire, DateofTermination) / 365.25 < 5  THEN '2-5 yr'
            WHEN DATEDIFF('day', DateofHire, DateofTermination) / 365.25 < 10 THEN '5-10 yr'
            ELSE '10+ yr'
        END AS TenureBand,
        COUNT(EmpID) AS count,
        ROUND(COUNT(EmpID)*100/SUM(COUNT(*)) OVER (), 2) AS pct
    FROM df
    WHERE EmploymentStatus != 'Future Start' AND Termd = 1
    GROUP BY TenureBand
    ORDER BY CASE TenureBand
        WHEN '0-1 yr'   THEN 1
        WHEN '1-2 yr'   THEN 2
        WHEN '2-5 yr'   THEN 3
        WHEN '5-10 yr'  THEN 4
        ELSE 5 END
""")

┌────────────┬───────┬────────┐
│ TenureBand │ count │  pct   │
│  varchar   │ int64 │ double │
├────────────┼───────┼────────┤
│ 0-1 yr     │    32 │  31.07 │
│ 1-2 yr     │    23 │  22.33 │
│ 2-5 yr     │    42 │  40.78 │
│ 5-10 yr    │     6 │   5.83 │
└────────────┴───────┴────────┘

### 5.7 — Demographics

In [27]:
# Gender attrition rates
# KEY FINDING: F 35.5% vs M 33.08% — no significant gender bias (2.4pp gap)
duckdb.sql("""
    SELECT Sex,
           COUNT(EmpID) AS total,
           COUNT(CASE WHEN Termd=1 THEN EmpID END) AS terminated,
           COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END) AS vol_terminated,
           COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END) AS invol_terminated,
           ROUND(COUNT(CASE WHEN Termd=1 THEN EmpID END)/COUNT(EmpID)*100, 2) AS overall_attrition_rate,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END)/COUNT(EmpID)*100, 2) AS vol_attrition_rate,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END)/COUNT(EmpID)*100, 2) AS invol_attrition_rate
    FROM df
    WHERE EmploymentStatus != 'Future Start'
    GROUP BY Sex
""")

┌─────────┬───────┬────────────┬────────────────┬──────────────────┬────────────────────────┬────────────────────┬──────────────────────┐
│   Sex   │ total │ terminated │ vol_terminated │ invol_terminated │ overall_attrition_rate │ vol_attrition_rate │ invol_attrition_rate │
│ varchar │ int64 │   int64    │     int64      │      int64       │         double         │       double       │        double        │
├─────────┼───────┼────────────┼────────────────┼──────────────────┼────────────────────────┼────────────────────┼──────────────────────┤
│ F       │   169 │         60 │             51 │                9 │                   35.5 │              30.18 │                 5.33 │
│ M       │   130 │         43 │             37 │                6 │                  33.08 │              28.46 │                 4.62 │
└─────────┴───────┴────────────┴────────────────┴──────────────────┴────────────────────────┴────────────────────┴──────────────────────┘

In [28]:
# Age band attrition rates — CORRECTED after DOB century fix
# Reference date: 2019-02-28 (last performance review date = data snapshot point)
# NOTE: Original analysis before the DOB fix incorrectly showed 98 employees "Under 25"
# (all were misparsed dates with negative ages) and "no employees over 45".
# After the fix the true age distribution includes the 45+ band.
duckdb.sql("""
    SELECT
        CASE
            WHEN DATEDIFF('day', DOB, DATE '2019-02-28') / 365.25 < 25 THEN 'Under 25'
            WHEN DATEDIFF('day', DOB, DATE '2019-02-28') / 365.25 < 35 THEN '25-34'
            WHEN DATEDIFF('day', DOB, DATE '2019-02-28') / 365.25 < 45 THEN '35-44'
            ELSE '45+'
        END AS AgeBand,
        COUNT(EmpID) AS total,
        COUNT(CASE WHEN Termd = 1 THEN EmpID END) AS terminated,
        ROUND(COUNT(CASE WHEN Termd = 1 THEN EmpID END) * 100.0 / COUNT(EmpID), 2) AS attrition_rate
    FROM df
    WHERE EmploymentStatus != 'Future Start'
    GROUP BY AgeBand
    ORDER BY CASE AgeBand
        WHEN 'Under 25' THEN 1
        WHEN '25-34'    THEN 2
        WHEN '35-44'    THEN 3
        ELSE 4 END
""").show()

┌─────────┬───────┬────────────┬────────────────┐
│ AgeBand │ total │ terminated │ attrition_rate │
│ varchar │ int64 │   int64    │     double     │
├─────────┼───────┼────────────┼────────────────┤
│ 25-34   │   102 │         34 │          33.33 │
│ 35-44   │   117 │         34 │          29.06 │
│ 45+     │    80 │         35 │          43.75 │
└─────────┴───────┴────────────┴────────────────┘



### 5.8 — Manager Analysis

In [29]:
# Manager-level attrition rates
# Managers with <10 direct reports excluded from comparisons (small sample)
# KEY FINDINGS:
# Webster Butler  68.42% — all voluntary exits
# Amy Dunn        61.90% — all voluntary exits
# Simon Roup      50.00% — equal voluntary/involuntary split (team dysfunction signal)
duckdb.sql("""
    SELECT ManagerName,
           COUNT(EmpID) AS total,
           COUNT(CASE WHEN Termd=1 THEN EmpID END) AS terminated,
           ROUND(COUNT(CASE WHEN Termd=1 THEN EmpID END)/COUNT(EmpID)*100, 2) AS overall_attrition_rate,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Voluntarily Terminated' THEN EmpID END)/COUNT(EmpID)*100, 2) AS vol_attrition_rate,
           ROUND(COUNT(CASE WHEN EmploymentStatus='Terminated for Cause' THEN EmpID END)/COUNT(EmpID)*100, 2) AS invol_attrition_rate
    FROM df
    WHERE EmploymentStatus != 'Future Start'
    GROUP BY ManagerName
    ORDER BY overall_attrition_rate DESC
""")

┌────────────────────┬───────┬────────────┬────────────────────────┬────────────────────┬──────────────────────┐
│    ManagerName     │ total │ terminated │ overall_attrition_rate │ vol_attrition_rate │ invol_attrition_rate │
│      varchar       │ int64 │   int64    │         double         │       double       │        double        │
├────────────────────┼───────┼────────────┼────────────────────────┼────────────────────┼──────────────────────┤
│ Webster Butler     │    19 │         13 │                  68.42 │              68.42 │                  0.0 │
│ Amy Dunn           │    21 │         13 │                   61.9 │               61.9 │                  0.0 │
│ Kissy Sullivan     │    22 │         12 │                  54.55 │              45.45 │                 9.09 │
│ Simon Roup         │    16 │          8 │                   50.0 │               25.0 │                 25.0 │
│ Michael Albert     │    20 │          9 │                   45.0 │               40.0 │       

---
## Section 6 — Master Table Build

Joins all four files into one enriched table for Power BI and the cost engine.

**Join sequence:**
- `df` (base) LEFT JOIN `salary` on `Position` → adds salary band columns
- `df` LEFT JOIN `FinalCostPerHire` CTE on `RecruitmentSource` → adds cost per hire

**Derived columns added:**

| Column | Formula | Notes |
|--------|---------|-------|
| `AnnualSalary` | PayRate × 40 × 52 | v13 uses hourly PayRate, not annual salary |
| `CompaRatio` | PayRate / HourlyMid | Hourly vs hourly — like for like |
| `PayPosition` | vs HourlyMin/Max | Below Market / At Market / Above Market / No Grid Data |
| `SalaryGapCost` | (HourlyMid - PayRate) × 40 × 52 | Annual cost to bring below-market employee to midpoint |
| `TenureYears` | DATEDIFF / 365.25 | Reference date: 2019-02-28 for active employees |
| `TermType` | from EmploymentStatus | Voluntary / Involuntary / Active |
| `TotalReplacementCost` | CostPerHire + SalaryGapCost | Terminated employees only; NULL for active |

**Output:** 299 rows × 49 columns (Future Start employees excluded)  
**Known gap:** Indeed has no cost record → 8 employees assigned average cost per hire (\$998.69)


In [30]:
# Check join key compatibility before building master table
print("=== Positions in HR not in salary grid (will produce No Grid Data) ===")
print(set(df['Position'].unique()) - set(salary['Position'].unique()))

print("\n=== Sources in HR not in recruiting costs (will produce NULL cost) ===")
print(set(df['RecruitmentSource'].unique()) - set(recruit['EmploymentSource'].unique()))

=== Positions in HR not in salary grid (will produce No Grid Data) ===
{'President & CEO', 'IT Director', 'Enterprise Architect', 'IT Support', 'Data Analyst', 'Software Engineer', 'Area Sales Manager', 'BI Developer', 'BI Director', 'IT Manager - DB', 'Director of Operations', 'Senior BI Developer', 'Data Architect', 'Shared Services Manager', 'Sales Manager', 'IT Manager - Support', 'Production Manager', 'Director of Sales', 'IT Manager - Infra', 'Software Engineering Manager', 'Principal Data Architect', 'CIO'}

=== Sources in HR not in recruiting costs (will produce NULL cost) ===
{'Indeed'}


In [31]:
# Build master table
# Cost per hire CTE: aggregates spend per source, counts hires, calculates cost
# Indeed employees (8) get average cost from paid sources ($998.69) via COALESCE

duckdb.sql("""
    CREATE OR REPLACE TABLE hr_master AS

    WITH source_spend AS (
        SELECT EmploymentSource, SUM(Cost) AS TotalSpend
        FROM recruit
        GROUP BY EmploymentSource
    ),
    hire_counts AS (
        SELECT RecruitmentSource, COUNT(EmpID) AS HireCount
        FROM df
        WHERE EmploymentStatus != 'Future Start'
        GROUP BY RecruitmentSource
    ),
    cost_per_hire AS (
        SELECT
            h.RecruitmentSource,
            s.TotalSpend,
            h.HireCount,
            ROUND(s.TotalSpend / h.HireCount, 2) AS CostPerHire
        FROM hire_counts h
        LEFT JOIN source_spend s ON h.RecruitmentSource = s.EmploymentSource
    ),
    FinalCostPerHire AS (
        SELECT
            RecruitmentSource,
            HireCount,
            COALESCE(
                CostPerHire,
                (SELECT ROUND(AVG(CostPerHire), 2) FROM cost_per_hire WHERE TotalSpend > 0)
            ) AS Indeed_fix
        FROM cost_per_hire
    )

    SELECT
        df.*,
        FinalCostPerHire.Indeed_fix                                         AS CostPerHire,
        salary.HourlyMin, salary.HourlyMid, salary.HourlyMax,
        salary.AnnualMin, salary.AnnualMid, salary.AnnualMax,

        ROUND(PayRate * 40 * 52, 2)                                         AS AnnualSalary,

        ROUND(PayRate / salary.HourlyMid, 4)                                AS CompaRatio,

        CASE
            WHEN salary.HourlyMid IS NULL   THEN 'No Grid Data'
            WHEN PayRate < salary.HourlyMin THEN 'Below Market'
            WHEN PayRate > salary.HourlyMax THEN 'Above Market'
            ELSE 'At Market'
        END                                                                  AS PayPosition,

        CASE
            WHEN salary.HourlyMin IS NOT NULL AND PayRate < salary.HourlyMin
            THEN ROUND((salary.HourlyMid - PayRate) * 40 * 52, 2)
            ELSE 0
        END                                                                  AS SalaryGapCost,

        ROUND(DATEDIFF('day', DateofHire,
            CASE WHEN Termd = 1 THEN DateofTermination ELSE DATE '2019-02-28' END
        ) / 365.25, 2)                                                       AS TenureYears,

        CASE
            WHEN EmploymentStatus = 'Voluntarily Terminated' THEN 'Voluntary'
            WHEN EmploymentStatus = 'Terminated for Cause'   THEN 'Involuntary'
            WHEN EmploymentStatus = 'Active'                 THEN 'Active'
            ELSE EmploymentStatus
        END                                                                  AS TermType,

        CASE
            WHEN Termd = 1
            THEN ROUND(COALESCE(FinalCostPerHire.Indeed_fix, 0) + COALESCE(SalaryGapCost, 0), 2)
            ELSE NULL
        END                                                                  AS TotalReplacementCost

    FROM df
    LEFT JOIN salary           ON df.Position         = salary.Position
    LEFT JOIN FinalCostPerHire ON df.RecruitmentSource = FinalCostPerHire.RecruitmentSource
    WHERE df.EmploymentStatus != 'Future Start'
""")

# Verify row count — must be 299
duckdb.sql("SELECT COUNT(*) AS total_rows FROM hr_master").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│        299 │
└────────────┘



In [32]:
# Pay equity check — how many employees are below market?
# KEY FINDING: 5 employees below market, all Production Technician I at $14.00/hr vs $14.42 minimum
duckdb.sql("""
    SELECT Employee_Name, Department, Position,
           PayRate, HourlyMin, HourlyMid,
           SalaryGapCost, TermType
    FROM hr_master
    WHERE SalaryGapCost > 0
    ORDER BY SalaryGapCost DESC
""").show()

┌─────────────────────┬────────────┬─────────────────────────┬─────────┬───────────┬───────────┬───────────────┬───────────┐
│    Employee_Name    │ Department │        Position         │ PayRate │ HourlyMin │ HourlyMid │ SalaryGapCost │ TermType  │
│       varchar       │  varchar   │         varchar         │ double  │  double   │  double   │    double     │  varchar  │
├─────────────────────┼────────────┼─────────────────────────┼─────────┼───────────┼───────────┼───────────────┼───────────┤
│ Sutwell, Barbara    │ Production │ Production Technician I │    14.0 │     14.42 │     19.23 │       10878.4 │ Active    │
│ Gross, Paula        │ Production │ Production Technician I │    14.0 │     14.42 │     19.23 │       10878.4 │ Voluntary │
│ Meads, Elizabeth    │ Production │ Production Technician I │    14.0 │     14.42 │     19.23 │       10878.4 │ Voluntary │
│ Knapp, Bradley  J   │ Production │ Production Technician I │    14.0 │     14.42 │     19.23 │       10878.4 │ Active    │


In [33]:
# Export master table to CSV for Power BI
duckdb.sql("COPY hr_master TO 'C:/Ongoing course/Projects/HR Attrition Analytics/hr_master.csv' (HEADER, DELIMITER ',')")
print("hr_master.csv exported successfully")

hr_master.csv exported successfully


---
## Section 7 — Cost Engine

Quantifies the financial impact of attrition using actual recruiting cost data.

**Cost model components:**

| Component | Source | Notes |
|-----------|--------|-------|
| `TotalReplacementCost` | CostPerHire + SalaryGapCost | In hr_master — direct recruiting spend |
| `VacancyLoss` | (AnnualSalary / 365) × 45 days | Productivity lost while role sits vacant |
| `Full Cost` | TotalReplacementCost + VacancyLoss | True total cost per exit |

**Key findings:**
- Total attrition cost: **\$762,620**
- Vacancy loss = 93.5% of total (\$713,276) — dominant cost driver
- Recruiting spend = only 6.5% (\$49,344) — the visible cost is a tiny fraction of the true cost
- Average cost per exit: **\$7,404**


In [34]:
# Total attrition cost including vacancy loss (45-day default)
duckdb.sql("""
    WITH
    exit_cost AS (
        SELECT
            ROUND(45 * SUM(AnnualSalary) / 365, 2) AS VacancyLoss,
            ROUND(SUM(TotalReplacementCost) + 45 * SUM(AnnualSalary) / 365, 2) AS Total_cost
        FROM hr_master
        WHERE Termd = 1
    ),
    exit_count AS (
        SELECT COUNT(*) AS terminated_count
        FROM hr_master
        WHERE Termd = 1
    )
    SELECT VacancyLoss, Total_cost, terminated_count,
           ROUND(Total_cost / terminated_count, 2) AS avg_cost_per_exit
    FROM exit_cost, exit_count
""")

┌─────────────┬────────────┬──────────────────┬───────────────────┐
│ VacancyLoss │ Total_cost │ terminated_count │ avg_cost_per_exit │
│   double    │   double   │      int64       │      double       │
├─────────────┼────────────┼──────────────────┼───────────────────┤
│   713275.59 │  762619.96 │              103 │           7404.08 │
└─────────────┴────────────┴──────────────────┴───────────────────┘

In [35]:
# Attrition cost breakdown by department
# KEY FINDINGS:
# Production: $523,744 total (68.7% of company total) — most absolute cost
# Sales: $14,884 avg per exit — most expensive per person
# Software Engineering: $12,862 avg per exit
duckdb.sql("""
    WITH
    exit_cost AS (
        SELECT Department,
               ROUND(45 * SUM(AnnualSalary) / 365, 2) AS VacancyLoss,
               ROUND(SUM(TotalReplacementCost) + 45 * SUM(AnnualSalary) / 365, 2) AS Total_cost
        FROM hr_master
        WHERE Termd = 1
        GROUP BY Department
    ),
    exit_count AS (
        SELECT Department, COUNT(*) AS terminated_count
        FROM hr_master
        WHERE Termd = 1
        GROUP BY Department
    )
    SELECT exit_cost.Department, VacancyLoss, Total_cost, terminated_count,
           ROUND(Total_cost / terminated_count, 2) AS avg_cost_per_exit
    FROM exit_cost
    JOIN exit_count ON exit_cost.Department = exit_count.Department
    ORDER BY Total_cost DESC
""")

┌──────────────────────┬─────────────┬────────────┬──────────────────┬───────────────────┐
│      Department      │ VacancyLoss │ Total_cost │ terminated_count │ avg_cost_per_exit │
│       varchar        │   double    │   double   │      int64       │      double       │
├──────────────────────┼─────────────┼────────────┼──────────────────┼───────────────────┤
│ Production           │   478962.74 │  523744.73 │               83 │           6310.18 │
│ IT/IS                │    114192.0 │  114997.21 │               10 │          11499.72 │
│ Sales                │    57762.74 │   59534.21 │                4 │          14883.55 │
│ Software Engineering │    37483.59 │   38586.37 │                3 │          12862.12 │
│ Admin Offices        │    24874.52 │   25757.44 │                3 │           8585.81 │
└──────────────────────┴─────────────┴────────────┴──────────────────┴───────────────────┘

In [36]:
# Top 10 most expensive individual departures
# KEY FINDING: Meads and Gross (Production Technician I, below-market pay)
# appear in top 10 despite $29K salary — SalaryGapCost inflates replacement cost
# Cost to replace: ~$14,800 each vs $873/yr to fix pay to minimum band
duckdb.sql("""
    SELECT Employee_Name, Department, Position, AnnualSalary,
           TotalReplacementCost, TermReason, TenureYears,
           ROUND(45 * AnnualSalary / 365, 2) AS VacancyLoss,
           ROUND(TotalReplacementCost + 45 * AnnualSalary / 365, 2) AS Total_cost
    FROM hr_master
    WHERE Termd = 1
    ORDER BY Total_cost DESC
    LIMIT 10
""").show()

┌───────────────────────┬───────────────┬──────────────────────────┬──────────────┬──────────────────────┬──────────────────────────────────┬─────────────┬─────────────┬────────────┐
│     Employee_Name     │  Department   │         Position         │ AnnualSalary │ TotalReplacementCost │            TermReason            │ TenureYears │ VacancyLoss │ Total_cost │
│        varchar        │    varchar    │         varchar          │    double    │        double        │             varchar              │   double    │   double    │   double   │
├───────────────────────┼───────────────┼──────────────────────────┼──────────────┼──────────────────────┼──────────────────────────────────┼─────────────┼─────────────┼────────────┤
│ Kampew, Donysha       │ Sales         │ Sales Manager            │     125320.0 │               506.64 │ maternity leave - did not return │        2.46 │    15450.41 │   15957.05 │
│ Ait Sidi, Karthikeyan │ IT/IS         │ Sr. DBA                  │     128960.0 │  

---
## Section 8 — Risk Segmentation & Export

Builds a forward-looking flight risk score for all **active employees** (196 total).

**Flight Risk Score components (0–100 scale, higher = higher risk):**

| Component | Weight | Formula |
|-----------|--------|---------|
| Engagement | 50% | (5 - EngagementSurvey) / 4 × 100 |
| Satisfaction | 30% | (5 - EmpSatisfaction) / 4 × 100 |
| CompaRatio | 20% | GREATEST(0, 1 - CompaRatio) × 100 — capped at 0 if above market |

**Note:** DaysLateLast30 removed — all zeros in v13, no analytical variance.

**Quadrant boundaries:** median FlightRiskScore (x-axis) and median EstimatedReplacementCost (y-axis)

| Quadrant | Condition | Count | Action |
|----------|-----------|-------|--------|
| Critical Risk | High risk + High cost | 47 (24%) | Immediate intervention |
| Manageable Risk | Low risk + High cost | 51 (26%) | Monitor engagement |
| High Flight Risk | High risk + Low cost | 50 (25.5%) | Lower-cost intervention |
| Low Priority | Low risk + Low cost | 48 (24.5%) | Standard management |

**Total Critical Risk replacement exposure: \$561,565**


In [37]:
# Load hr_master from CSV for risk scoring
# (reload to ensure we're working from the clean exported file)
hr_master = pd.read_csv(r'C:\Ongoing course\Projects\HR Attrition Analytics\hr_master.csv')
print("hr_master loaded:", hr_master.shape)

hr_master loaded: (299, 49)


In [38]:
# Build risk segments table and export
# Active employees only (Termd=0, excluding Future Start)
duckdb.sql("""
    CREATE OR REPLACE TABLE risk_segments AS

    WITH scores AS (
        SELECT
            Employee_Name,
            ROUND((5 - EngagementSurvey) / 4 * 100, 2)                          AS EngagementScore,
            ROUND((5 - EmpSatisfaction)  / 4 * 100, 2)                          AS SatisfactionScore,
            ROUND(GREATEST(0, 1 - CompaRatio) * 100, 2)                         AS CompaRatioScore,
            ROUND(
                (5 - EngagementSurvey) / 4 * 100 * 0.50 +
                (5 - EmpSatisfaction)  / 4 * 100 * 0.30 +
                GREATEST(0, 1 - CompaRatio) * 100 * 0.20, 2
            )                                                                     AS FlightRiskScore,
            ROUND(45 * AnnualSalary / 365, 2)                                    AS VacancyLoss,
            ROUND(CostPerHire + 45 * AnnualSalary / 365, 2)                      AS EstimatedReplacementCost
        FROM hr_master
        WHERE Termd = 0 AND EmploymentStatus != 'Future Start'
    ),
    medians AS (
        SELECT
            MEDIAN(FlightRiskScore)          AS median_risk,
            MEDIAN(EstimatedReplacementCost) AS median_cost
        FROM scores
    )

    SELECT
        scores.*,
        median_risk,
        median_cost,
        CASE
            WHEN FlightRiskScore > median_risk AND EstimatedReplacementCost > median_cost THEN 'Critical Risk'
            WHEN FlightRiskScore <= median_risk AND EstimatedReplacementCost > median_cost THEN 'Manageable Risk'
            WHEN FlightRiskScore > median_risk AND EstimatedReplacementCost <= median_cost THEN 'High Flight Risk'
            ELSE 'Low Priority'
        END AS RiskQuadrant
    FROM scores, medians
    ORDER BY FlightRiskScore DESC
""")

duckdb.sql("COPY risk_segments TO 'C:/Ongoing course/Projects/HR Attrition Analytics/risk_segments.csv' (HEADER, DELIMITER ',')")
print("risk_segments.csv exported successfully")

risk_segments.csv exported successfully


In [39]:
# Quadrant distribution — should be approximately 25% each (median-based split)
duckdb.sql("""
    SELECT RiskQuadrant,
           COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / 196, 1) AS pct
    FROM risk_segments
    GROUP BY RiskQuadrant
    ORDER BY count DESC
""").show()

┌──────────────────┬───────┬────────┐
│   RiskQuadrant   │ count │  pct   │
│     varchar      │ int64 │ double │
├──────────────────┼───────┼────────┤
│ Manageable Risk  │    51 │   26.0 │
│ High Flight Risk │    50 │   25.5 │
│ Low Priority     │    48 │   24.5 │
│ Critical Risk    │    47 │   24.0 │
└──────────────────┴───────┴────────┘



In [40]:
# Critical Risk total replacement exposure — input to Power BI ROI Calculator
# KEY FINDING: $561,565 at risk if all 47 Critical Risk employees leave
duckdb.sql("""
    SELECT ROUND(SUM(EstimatedReplacementCost), 2) AS critical_risk_total_cost
    FROM risk_segments
    WHERE RiskQuadrant = 'Critical Risk'
""").show()

┌──────────────────────────┐
│ critical_risk_total_cost │
│          double          │
├──────────────────────────┤
│                561564.76 │
└──────────────────────────┘



---
## Export Summary

| File | Rows | Columns | Destination |
|------|------|---------|-------------|
| `hr_master.csv` | 299 | 49 | Power BI — main data table |
| `risk_segments.csv` | 196 | 10 | Power BI — risk quadrant scatter chart + ROI Calculator |


